# Semantic Search with FAISS and Sentence Transformers

This notebook demonstrates semantic search over a collection of text documents using:
- **FAISS HNSW** index with cosine similarity
- **Sentence Transformers** for embeddings
- **Local text files** as the data source

## Quick Start

1. Place your `.txt` files in a `documents/` folder
2. Run all cells
3. Search using the interactive widget at the bottom

**Requirements**: Works in SageMaker, Jupyter, Colab, or any notebook environment.

## 1. Install Dependencies

Run this cell first to install required packages.

In [ ]:
!pip install -q sentence-transformers faiss-cpu numpy tqdm

## 2. Import Libraries

In [ ]:
import os
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple
import json

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

print("✓ Libraries imported successfully")

## 3. Configuration

Adjust these settings as needed.

In [ ]:
# Configuration
DOCUMENTS_DIR = "documents"  # Folder containing your .txt files
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # Fast, good quality
CHUNK_SIZE = 500  # Characters per chunk
CHUNK_OVERLAP = 100  # Overlap between chunks
TOP_K = 5  # Number of results to return

# HNSW parameters
HNSW_M = 32  # Connections per node
HNSW_EF_CONSTRUCTION = 64  # Build quality
HNSW_EF_SEARCH = 64  # Search quality

print(f"Configuration:")
print(f"  Documents folder: {DOCUMENTS_DIR}")
print(f"  Model: {MODEL_NAME}")
print(f"  Chunk size: {CHUNK_SIZE} chars")
print(f"  Top-K results: {TOP_K}")

## 4. Data Structures

In [ ]:
@dataclass
class Document:
    """Represents a single document."""
    filename: str
    content: str
    
@dataclass
class Chunk:
    """Represents a chunk of a document."""
    doc_id: int
    chunk_id: int
    text: str
    filename: str

print("✓ Data structures defined")

## 5. Load Documents from Folder

This cell reads all `.txt` files from the documents folder.

In [ ]:
def load_documents(folder_path: str) -> List[Document]:
    """Load all .txt files from a folder."""
    docs_path = Path(folder_path)
    
    # Create folder if it doesn't exist
    docs_path.mkdir(exist_ok=True)
    
    documents = []
    txt_files = list(docs_path.glob("*.txt"))
    
    if not txt_files:
        print(f"⚠️  No .txt files found in '{folder_path}/'")
        print(f"   Please add some .txt files to the folder and re-run this cell.")
        return documents
    
    for filepath in tqdm(txt_files, desc="Loading documents"):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
            documents.append(Document(
                filename=filepath.name,
                content=content
            ))
        except Exception as e:
            print(f"Error loading {filepath.name}: {e}")
    
    return documents

# Load documents
documents = load_documents(DOCUMENTS_DIR)
print(f"\n✓ Loaded {len(documents)} documents")
if documents:
    print(f"  Example: {documents[0].filename} ({len(documents[0].content)} chars)")

## 6. Chunk Documents

Split documents into overlapping chunks for better search precision.

In [ ]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    text_len = len(text)
    
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end]
        
        if chunk.strip():  # Only add non-empty chunks
            chunks.append(chunk)
        
        # Move forward by chunk_size - overlap
        start += chunk_size - overlap
        
        # Break if we're at the end
        if end >= text_len:
            break
    
    return chunks

def create_chunks(documents: List[Document], chunk_size: int, overlap: int) -> List[Chunk]:
    """Create chunks from all documents."""
    all_chunks = []
    
    for doc_id, doc in enumerate(tqdm(documents, desc="Chunking documents")):
        text_chunks = chunk_text(doc.content, chunk_size, overlap)
        
        for chunk_id, text in enumerate(text_chunks):
            all_chunks.append(Chunk(
                doc_id=doc_id,
                chunk_id=chunk_id,
                text=text,
                filename=doc.filename
            ))
    
    return all_chunks

# Create chunks
if documents:
    chunks = create_chunks(documents, CHUNK_SIZE, CHUNK_OVERLAP)
    print(f"\n✓ Created {len(chunks)} chunks from {len(documents)} documents")
    print(f"  Avg chunks per document: {len(chunks) / len(documents):.1f}")
else:
    chunks = []
    print("⚠️  No documents to chunk. Please add .txt files and re-run cells 5-6.")

## 7. Load Embedding Model

Download and load the sentence transformer model.

In [ ]:
print(f"Loading model: {MODEL_NAME}")
print("This may take a minute on first run (downloading model)...")

model = SentenceTransformer(MODEL_NAME, device="cpu")

print(f"\n✓ Model loaded successfully")
print(f"  Embedding dimension: {model.get_sentence_embedding_dimension()}")

## 8. Create Embeddings

Encode all chunks into vector embeddings.

In [ ]:
if chunks:
    print(f"Encoding {len(chunks)} chunks...")
    
    # Extract text from chunks
    chunk_texts = [chunk.text for chunk in chunks]
    
    # Encode with progress bar
    embeddings = model.encode(
        chunk_texts,
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True,  # For cosine similarity
        show_progress_bar=True
    )
    
    print(f"\n✓ Embeddings created")
    print(f"  Shape: {embeddings.shape}")
    print(f"  Memory: ~{embeddings.nbytes / 1024 / 1024:.1f} MB")
else:
    embeddings = None
    print("⚠️  No chunks to encode. Please add documents and re-run cells 5-8.")

## 9. Build FAISS HNSW Index

Create a fast HNSW index with cosine similarity.

In [ ]:
def create_hnsw_index(embeddings: np.ndarray, M: int, ef_construction: int, ef_search: int) -> faiss.Index:
    """Create FAISS HNSW index with cosine similarity."""
    embeddings = np.asarray(embeddings, dtype='float32')
    dim = embeddings.shape[1]
    
    # Create HNSW index with Inner Product (cosine for normalized vectors)
    index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
    
    # Set construction parameters
    index.hnsw.efConstruction = ef_construction
    
    # Add vectors
    index.add(embeddings)
    
    # Set search parameter
    index.hnsw.efSearch = ef_search
    
    return index

if embeddings is not None:
    print("Building FAISS HNSW index...")
    
    index = create_hnsw_index(
        embeddings,
        M=HNSW_M,
        ef_construction=HNSW_EF_CONSTRUCTION,
        ef_search=HNSW_EF_SEARCH
    )
    
    print(f"\n✓ HNSW index built successfully")
    print(f"  Index type: {type(index).__name__}")
    print(f"  Vectors indexed: {index.ntotal}")
    print(f"  Similarity metric: Cosine (via normalized Inner Product)")
    print(f"  HNSW M: {HNSW_M}")
    print(f"  efConstruction: {HNSW_EF_CONSTRUCTION}")
    print(f"  efSearch: {HNSW_EF_SEARCH}")
else:
    index = None
    print("⚠️  No embeddings to index. Please add documents and re-run cells 5-9.")

## 10. Search Function

Define the semantic search function.

In [ ]:
def search(query: str, top_k: int = 5) -> List[Tuple[float, Chunk]]:
    """Perform semantic search."""
    if index is None:
        print("⚠️  Index not built. Please add documents and re-run cells 5-9.")
        return []
    
    # Encode query
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    
    # Search
    scores, indices = index.search(query_embedding.astype(np.float32), top_k)
    
    # Prepare results
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(chunks):  # Valid index
            results.append((float(score), chunks[idx]))
    
    return results

def display_results(query: str, results: List[Tuple[float, Chunk]]):
    """Display search results in a nice format."""
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}\n")
    
    if not results:
        print("No results found.")
        return
    
    for rank, (score, chunk) in enumerate(results, 1):
        print(f"[{rank}] {chunk.filename} (chunk {chunk.chunk_id})")
        print(f"Score: {score:.3f}")
        print(f"Snippet: {chunk.text[:200]}..." if len(chunk.text) > 200 else f"Text: {chunk.text}")
        print(f"{'-'*80}\n")

print("✓ Search functions defined")

## 11. Example Searches

Try some example searches (customize based on your documents).

In [ ]:
# Example search 1
query1 = "What are the main topics?"
results1 = search(query1, top_k=TOP_K)
display_results(query1, results1)

In [ ]:
# Example search 2
query2 = "key findings and conclusions"
results2 = search(query2, top_k=TOP_K)
display_results(query2, results2)

## 12. Interactive Search

Run this cell to search interactively.

In [ ]:
def interactive_search():
    """Interactive search loop."""
    print("\n" + "="*80)
    print("Interactive Semantic Search")
    print("="*80)
    print("Enter your query (or 'quit' to exit)\n")
    
    while True:
        query = input("Query> ").strip()
        
        if query.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break
        
        if not query:
            continue
        
        results = search(query, top_k=TOP_K)
        display_results(query, results)

# Run interactive search
interactive_search()

## 13. Advanced: Adjust Parameters

Experiment with different search parameters.

In [ ]:
# Try with more results
query = "your search query here"
results = search(query, top_k=10)
display_results(query, results)

In [ ]:
# Adjust HNSW efSearch for speed/quality tradeoff
if index is not None:
    # Higher = more accurate, slower
    index.hnsw.efSearch = 128
    print(f"✓ Increased search quality (efSearch=128)")
    
    # Lower = faster, slightly less accurate
    # index.hnsw.efSearch = 32
    # print(f"✓ Increased search speed (efSearch=32)")

## 14. Export Results (Optional)

Save search results to a file.

In [ ]:
def export_results(query: str, results: List[Tuple[float, Chunk]], filename: str):
    """Export results to JSON file."""
    output = {
        "query": query,
        "results": [
            {
                "rank": rank,
                "score": float(score),
                "filename": chunk.filename,
                "chunk_id": chunk.chunk_id,
                "text": chunk.text
            }
            for rank, (score, chunk) in enumerate(results, 1)
        ]
    }
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Results exported to {filename}")

# Example: export results
# query = "your query"
# results = search(query)
# export_results(query, results, "search_results.json")

## Summary

### What This Notebook Does:

1. ✓ Loads text documents from `documents/` folder
2. ✓ Chunks documents into overlapping segments
3. ✓ Creates embeddings using sentence transformers
4. ✓ Builds FAISS HNSW index with cosine similarity
5. ✓ Provides semantic search functionality
6. ✓ Interactive search interface

### Usage Tips:

- **Add documents**: Place `.txt` files in the `documents/` folder
- **Adjust chunk size**: Modify `CHUNK_SIZE` for your use case
- **Change model**: Try `all-mpnet-base-v2` for better quality (slower)
- **Tune HNSW**: Adjust `M`, `efConstruction`, `efSearch` for speed/quality
- **More results**: Increase `TOP_K` or pass `top_k` to `search()`

### Next Steps:

- Experiment with different queries
- Try different chunking strategies
- Compare embedding models
- Add more documents and re-run cells 5-9

---

**Ready to use!** Run all cells and start searching. 🔍